# Issue #12: Feature Engineering

**Input:** `acn_clean_sessions.csv` (Issue #9)  
**Code:** `feature_engineering.py` (repository root). This notebook runs and documents it, so the features are identical whether built here or with `python feature_engineering.py`.  
**Outputs (repository root):** `feature_engineered_sessions.csv`, `feature_engineered_daily.csv`, `feature_dictionary.csv`, `feature_validation_summary.csv`

**Design rules**
- **Calendar-based lags.** The daily table is reindexed to a complete calendar before any lag is computed, so `lag_7` is always the value 7 *calendar days* earlier. Days with no sessions are real zeros, except runs of 7+ zero days, which are **recording gaps**: their target is unknown (NaN), so no lag or rolling value is ever computed from them.
- **Roles.** Every daily column is tagged in `feature_dictionary.csv`. Only `role == "predictor"` columns may enter a model, and all of them are known before the forecast day starts.
- **Same-day aggregates** (average kWh, long-stay share, claimed share, …) describe the day being predicted. They keep a `same_day_` prefix for analysis and are never predictors.
- **Regime flags.** `is_r2_policy` (1 Sep 2018, 14 kWh default), `is_paid` (1 Nov 2018, paid charging) and `is_covid` (16 Mar 2020, remote operation; an assumption checked in Issue #10) let one model cover the whole period. Issues #10–#11 showed these changes move demand more than anything else.
- **Holidays.** `is_major_holiday` flags the campus closures found in the data (Issue #16, §1.6): New Year, MLK Day, Presidents' Day, Memorial Day, 4 July, Labor Day, Thanksgiving and the next day, and 24–31 December. Columbus Day, Veterans Day and Juneteenth behave like normal days at this site and are kept for reference only.
- **Excluded as predictors:** `month`, `day_of_month`, `week_of_year` (the only full year in one regime is 2019, so seasonality cannot be separated from regime changes) and the station statistics in the session table (computed on the full dataset, so they would leak test-period information).

In [1]:
from pathlib import Path
def find_root(marker):
    """Repository root = the folder holding `marker` (this folder or its parent)."""
    for p in [Path.cwd(), Path.cwd().parent]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"{marker} not found in this folder or its parent")
import sys
ROOT = find_root("acn_clean_sessions.csv")
sys.path.insert(0, str(ROOT))

import importlib
import pandas as pd
from IPython.display import display, Markdown
import feature_engineering as fe
importlib.reload(fe)

source = pd.read_csv(ROOT / "acn_clean_sessions.csv")
session_features = fe.build_session_features(source)
daily_features = fe.build_daily_features(session_features)
dictionary = fe.feature_dictionary(daily_features)
validation = fe.validate(source, session_features, daily_features)
display(validation)
assert validation["passed"].all(), "Feature validation failed"

session_features.to_csv(ROOT / "feature_engineered_sessions.csv", index=False)
daily_features.to_csv(ROOT / "feature_engineered_daily.csv", index=False)
dictionary.to_csv(ROOT / "feature_dictionary.csv", index=False)
validation.to_csv(ROOT / "feature_validation_summary.csv", index=False)
print(f"{len(session_features):,} session rows x {session_features.shape[1]} columns")
print(f"{len(daily_features):,} daily rows x {daily_features.shape[1]} columns "
      f"({daily_features['date'].min().date()} to {daily_features['date'].max().date()})")

,check,passed
0,session_row_count_matches_input,True
1,session_ids_unique,True
2,daily_calendar_contiguous,True
3,daily_dates_unique,True
4,daily_counts_sum_to_sessions,True
5,lag_1_is_previous_calendar_day,True
6,lag_7_is_seven_calendar_days_earlier,True
7,rolling_7_mean_uses_previous_7_days_only,True
8,no_target_on_recording_gap_days,True
9,no_lag_taken_from_a_gap_day,True


31,387 session rows x 59 columns
1,238 daily rows x 43 columns (2018-04-25 to 2021-09-13)


## Feature dictionary (daily table)

In [2]:
print(dictionary["role"].value_counts().to_string())
display(dictionary)

role
predictor           19
same_day_outcome    12
calendar_info        6
meta                 4
target               2


,column,role,description
0,date,meta,Calendar day (local time)
1,session_count,target,Sessions starting on the day (primary target);...
2,total_kWh,target,Energy delivered by sessions starting on the d...
3,same_day_avg_kWh,same_day_outcome,Aggregate of the day being predicted - analysi...
4,same_day_median_kWh,same_day_outcome,Aggregate of the day being predicted - analysi...
5,same_day_avg_connected_h,same_day_outcome,Aggregate of the day being predicted - analysi...
6,same_day_avg_charging_h,same_day_outcome,Aggregate of the day being predicted - analysi...
7,same_day_avg_idle_h,same_day_outcome,Aggregate of the day being predicted - analysi...
8,same_day_avg_power_kW,same_day_outcome,Aggregate of the day being predicted - analysi...
9,same_day_n_stations,same_day_outcome,Aggregate of the day being predicted - analysi...


## Days per regime, recording gaps and zero-session days

In [3]:
reg = daily_features.groupby("regime").agg(calendar_days=("date", "size"), first=("date", "min"), last=("date", "max"),
                                           gap_days=("in_collection_gap", "sum"),
                                           zero_session_days=("session_count", lambda s: int((s == 0).sum())),
                                           mean_sessions=("session_count", "mean")).reindex(fe.REGIMES)
display(reg)

,calendar_days,first,last,gap_days,zero_session_days,mean_sessions
regime,,,,,,
R1 free,129,2018-04-25,2018-08-31,0,0,64.813953
"R2 free, 14 kWh default",61,2018-09-01,2018-10-31,0,0,77.622951
R3 paid,501,2018-11-01,2020-03-15,0,1,29.544910
R4 COVID period,547,2020-03-16,2021-09-13,106,47,7.911565


## Predictors: coverage and warm-up

In [4]:
pred = daily_features[fe.PREDICTORS]
cov = pd.DataFrame({"non_null": pred.notna().sum(),
                    "first_available": [daily_features.loc[pred[c].notna(), "date"].min().date() for c in pred]})
display(cov)
usable = pred.notna().all(axis=1) & daily_features["session_count"].notna()
print(f"Days usable for modelling (target and every predictor available): {int(usable.sum())} of {len(daily_features)}")
print("Not usable: lag warm-up at the start of the data, the recording gap, and the first days after the gap "
      "(their lags would reach into the gap).")

,non_null,first_available
dayofweek,1238,2018-04-25
is_weekend,1238,2018-04-25
dow_1,1238,2018-04-25
dow_2,1238,2018-04-25
dow_3,1238,2018-04-25
dow_4,1238,2018-04-25
dow_5,1238,2018-04-25
dow_6,1238,2018-04-25
is_major_holiday,1238,2018-04-25
is_r2_policy,1238,2018-04-25


Days usable for modelling (target and every predictor available): 1104 of 1238
Not usable: lag warm-up at the start of the data, the recording gap, and the first days after the gap (their lags would reach into the gap).


## Holidays in the data

In [5]:
hol = daily_features.loc[daily_features["holiday_name"].fillna("").ne(""),
                         ["date", "holiday_name", "is_major_holiday", "regime", "session_count", "lag_7"]]
display(hol)

,date,holiday_name,is_major_holiday,regime,session_count,lag_7
33,2018-05-28,Memorial Day,1,R1 free,20.0,67.0
70,2018-07-04,Independence Day,1,R1 free,24.0,69.0
131,2018-09-03,Labor Day,1,"R2 free, 14 kWh default",34.0,91.0
166,2018-10-08,Columbus Day,0,"R2 free, 14 kWh default",87.0,92.0
201,2018-11-12,Veterans Day,0,R3 paid,54.0,69.0
211,2018-11-22,Thanksgiving Day,1,R3 paid,9.0,62.0
212,2018-11-23,Day after Thanksgiving,1,R3 paid,7.0,57.0
243,2018-12-24,Winter break (24-31 Dec),1,R3 paid,3.0,37.0
244,2018-12-25,Christmas Day,1,R3 paid,5.0,48.0
245,2018-12-26,Winter break (24-31 Dec),1,R3 paid,6.0,43.0


## Summary

`feature_engineering.py` builds the session table, the daily modelling table, the feature dictionary and the validation checks. The checks above confirm that lags refer to calendar days, rolling windows use only previous days, no value is taken from a recording-gap day, regime flags match their dates, no predictor is a same-day outcome, and daily counts add up to the session table. Issue #13 reads `feature_engineered_daily.csv` and the `PREDICTORS` list from the module, so modelling and feature engineering cannot drift apart.